In [1]:
# Instalar PyTorch con CUDA (primero, para evitar que sentence-transformers instale CPU)
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Core libs
%pip install matplotlib==3.9.0 \
datasets==2.20.0 pyarrow==15.0.2 \
lime==0.2.0.1 shap==0.45.1 scipy tqdm pandas

In [ ]:
import random

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset, Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.stats import spearmanr, pearsonr
from tqdm import tqdm

from lime.lime_text import LimeTextExplainer
import shap

In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [ ]:
dataset_raw = load_dataset("wangrongsheng/ag_news")

In [6]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [ ]:
CLASS_LABELS     = ["World", "Sports", "Business", "Science"]
label_token_dict = {0: "World", 1: "Sports", 2: "Business", 3: "Science"}

def build_prompt(text):
    return f"Article: {text}\n\nTheme:"

def build_full_text(text, label):
    return f"Article: {text}\n\nTheme: {label_token_dict[label]}"

def preprocess_dataset(examples, tokenizer):
    texts_list, ids_list, mask_list, lab_list, true_list = [], [], [], [], []

    for ex in examples:
        text  = ex["text"]
        label = ex["label"]

        full_enc   = tokenizer(build_full_text(text, label), add_special_tokens=False)
        prompt_enc = tokenizer(build_prompt(text),           add_special_tokens=False)

        input_ids      = full_enc["input_ids"]
        attention_mask = [1] * len(input_ids)
        prompt_len     = len(prompt_enc["input_ids"])
        labs           = [-100 if i < prompt_len else tok for i, tok in enumerate(input_ids)]

        if len(input_ids) > MAX_LENGTH:
            input_ids      = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labs           = labs[:MAX_LENGTH]
        else:
            pad_len         = MAX_LENGTH - len(input_ids)
            input_ids      += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labs           += [-100] * pad_len

        texts_list.append(text)
        ids_list.append(input_ids)
        mask_list.append(attention_mask)
        lab_list.append(labs)
        true_list.append(label_token_dict[label])

    return Dataset.from_dict({
        "texts":          texts_list,
        "input_ids":      ids_list,
        "attention_mask": mask_list,
        "labels":         lab_list,
        "true_label":     true_list,
    })

In [8]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def fits_model(example):
    text = build_full_text(example["text"], example["label"])
    return len(tokenizer(text, add_special_tokens=False)["input_ids"]) <= MAX_MODEL_LENGTH

In [ ]:
BATCH_SIZE = 1

df_test       = pd.DataFrame(dataset_raw["test"])
filtered_test = df_test[df_test.apply(fits_model, axis=1)]
test_records  = filtered_test.to_dict("records")
test_dataset  = preprocess_dataset(test_records, tokenizer)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Test dataset: {len(test_dataset)} examples")

In [11]:
test_dataset[0]

{'texts': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as

In [ ]:
CHECKPOINT_DIR = "../checkpoints/agnews"

config_11 = AutoConfig.from_pretrained(MODEL_NAME, local_files_only=True)
config_11.num_hidden_layers = 11

def load_model(num_layers, ckpt_name):
    if num_layers == 22:
        m = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, local_files_only=True, attn_implementation="eager"
        )
    else:
        m = AutoModelForCausalLM.from_config(config_11)
    m.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/{ckpt_name}", map_location="cpu"))
    m.eval()
    return m

MODELS = {
    "teacher":       load_model(22, "best_teacher_model.pt"),
    "baseline":      load_model(11, "best_baseline_model.pt"),
    "bad_student":   load_model(11, "best_bad_student_model.pt"),
    "student":       load_model(11, "best_student_model.pt"),
    "student_local": load_model(11, "best_student_local_model.pt"),
}
MODEL_LABELS = {
    "teacher":       "P (teacher)",
    "baseline":      "B (baseline, no KD)",
    "bad_student":   "S_bad",
    "student":       "S1",
    "student_local": "S2",
}
print("Models loaded:", list(MODELS.keys()))

In [ ]:
class_token_ids = [
    tokenizer.encode(f" {lbl}", add_special_tokens=False)[0]
    for lbl in CLASS_LABELS
]
CLASS_LABEL_TO_IDX = {lbl: i for i, lbl in enumerate(CLASS_LABELS)}

def make_prompt(text):
    return f"Article: {text}\n\nTheme:"

def classify_fn(texts, model, batch_size=8):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    texts = [str(t) for t in texts]
    prompts = [make_prompt(t) for t in texts]
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i + batch_size]
            enc = tokenizer(batch, return_tensors="pt", truncation=True, padding=True).to(device)
            logits = model(**enc).logits
            last_idx = enc["attention_mask"].sum(dim=1) - 1
            logits = logits[torch.arange(len(batch), device=device), last_idx]
            class_logits = logits[:, class_token_ids]
            probs = F.softmax(class_logits, dim=-1)
            all_probs.append(probs.float().cpu().numpy())
    return np.concatenate(all_probs, axis=0)

# ── Evaluation subset: N=100 (25 per class, fixed seed=42) ───────────────
random.seed(42)
class_indices = {lbl: [] for lbl in CLASS_LABELS}
for i in range(len(test_dataset)):
    lbl = test_dataset[i]["true_label"]
    class_indices[lbl].append(i)

eval_indices = []
for lbl in CLASS_LABELS:
    eval_indices += random.sample(class_indices[lbl], 25)

eval_texts         = [test_dataset[i]["texts"] for i in eval_indices]
eval_labels        = [test_dataset[i]["true_label"] for i in eval_indices]
eval_class_indices = [CLASS_LABEL_TO_IDX[lbl] for lbl in eval_labels]

QUAL_EXAMPLES = {lbl: test_dataset[class_indices[lbl][0]]["texts"] for lbl in CLASS_LABELS}

print(f"Eval subset: {len(eval_texts)} examples")
for lbl in CLASS_LABELS:
    print(f"  {lbl}: {eval_labels.count(lbl)}")

# Qualitative Analysis

In [ ]:
def tokens_to_words(tokens, scores):
    words, word_scores = [], []
    current_word, current_scores = "", []
    for token, score in zip(tokens, scores):
        if token.startswith("▁") or token.startswith("Ġ"):
            if current_word:
                words.append(current_word)
                word_scores.append(np.mean(current_scores))
            current_word   = token.lstrip("▁").lstrip("Ġ")
            current_scores = [score]
        else:
            current_word  += token
            current_scores.append(score)
    if current_word:
        words.append(current_word)
        word_scores.append(np.mean(current_scores))
    return words, np.array(word_scores)

def integrated_gradients(model, input_ids, target_token_idx, steps=50):
    emb      = model.get_input_embeddings()(input_ids)
    baseline = torch.zeros_like(emb)
    grads    = []
    for alpha in torch.linspace(0, 1, steps):
        interp = (baseline + alpha * (emb - baseline)).detach().requires_grad_(True)
        score  = model(inputs_embeds=interp).logits[0, -1, target_token_idx]
        score.backward()
        grads.append(interp.grad.clone())
    avg_grads = torch.stack(grads).mean(dim=0)
    ig        = (emb - baseline) * avg_grads
    return ig.sum(dim=-1).squeeze(0)

def get_ig_scores(model, text, target_id):
    prompt = make_prompt(text)
    enc    = tokenizer(prompt, return_tensors="pt").to(device)
    ig     = integrated_gradients(model, enc["input_ids"], target_id)

    prefix_ids = tokenizer("Article: ", add_special_tokens=False)["input_ids"]
    text_ids   = tokenizer(text, add_special_tokens=False)["input_ids"]
    start = len(prefix_ids) + 1
    end   = start + len(text_ids)

    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    words, wscores = tokens_to_words(
        toks[start:end],
        ig.float().detach().cpu().numpy()[start:end]
    )
    return dict(zip(words, wscores.tolist()))

def get_lime_scores(model, text, class_idx=0, num_samples=2000, seed=42):
    explainer = LimeTextExplainer(class_names=CLASS_LABELS, random_state=seed)
    exp = explainer.explain_instance(
        text,
        lambda t, m=model: classify_fn(t, model=m),
        num_samples=num_samples,
        num_features=len(text.split()),
        labels=[class_idx],
    )
    return dict(exp.as_list(label=class_idx))

def get_shap_scores(model, text, class_idx=0):
    explainer = shap.Explainer(
        lambda t, m=model: classify_fn(t, model=m),
        shap.maskers.Text()
    )
    sv     = explainer([text])
    words  = list(sv.data[0])
    values = sv.values[0, :, class_idx].tolist()
    return dict(zip(words, values))

## LIME

In [ ]:
# LIME qualitative — one example per class, all 5 models
for lbl in CLASS_LABELS:
    class_idx = CLASS_LABEL_TO_IDX[lbl]
    text      = QUAL_EXAMPLES[lbl]
    print(f"\n{'='*60}\nClass: {lbl}\n{'='*60}")
    print(f"Text: {text[:120]}...")
    for name, model in MODELS.items():
        model.to(device)
        exp = LimeTextExplainer(class_names=CLASS_LABELS).explain_instance(
            text,
            lambda t, m=model: classify_fn(t, model=m),
            num_samples=5000,
            labels=[class_idx],
        )
        fig = exp.as_pyplot_figure(label=class_idx)
        fig.suptitle(f"LIME  ·  {MODEL_LABELS[name]}  ·  {lbl}", fontsize=11)
        plt.tight_layout()
        plt.savefig(f"../images/{name}_lime_{lbl.lower()}.png", dpi=150, bbox_inches="tight")
        plt.show()
        model.cpu()
        torch.cuda.empty_cache()

## SHAP

In [ ]:
# SHAP qualitative — one example per class, all 5 models
for lbl in CLASS_LABELS:
    class_idx = CLASS_LABEL_TO_IDX[lbl]
    text      = QUAL_EXAMPLES[lbl]
    print(f"\n{'='*60}\nClass: {lbl}\n{'='*60}")
    for name, model in MODELS.items():
        model.to(device)
        exp = shap.Explainer(
            lambda t, m=model: classify_fn(t, model=m),
            shap.maskers.Text()
        )
        sv = exp([text])
        print(f"\n{MODEL_LABELS[name]}")
        shap.plots.waterfall(sv[:, :, class_idx][0], show=False)
        plt.savefig(f"../images/{name}_shap_{lbl.lower()}.png", dpi=150, bbox_inches="tight")
        plt.show()
        model.cpu()
        torch.cuda.empty_cache()

## Integrated gradients

In [ ]:
# IG qualitative — one example per class, all 5 models
top_n = 15
for lbl in CLASS_LABELS:
    class_idx = CLASS_LABEL_TO_IDX[lbl]
    target_id = class_token_ids[class_idx]
    text      = QUAL_EXAMPLES[lbl]
    print(f"\n{'='*60}\nClass: {lbl}  |  target_id={target_id}\n{'='*60}")
    for name, model in MODELS.items():
        model.to(device)
        scores = get_ig_scores(model, text, target_id)
        model.cpu()
        torch.cuda.empty_cache()
        if not scores:
            continue
        words_list = list(scores.keys())
        vals       = np.array(list(scores.values()))
        top_idx    = np.argsort(np.abs(vals))[-top_n:][::-1]
        tw, ts     = [words_list[i] for i in top_idx], vals[top_idx]
        colors     = ["green" if s > 0 else "red" for s in ts]
        plt.figure(figsize=(10, 3))
        plt.bar(range(len(tw)), ts, color=colors)
        plt.xticks(range(len(tw)), tw, rotation=45, ha="right", fontsize=8)
        plt.title(f"IG  ·  {MODEL_LABELS[name]}  ·  {lbl}")
        plt.tight_layout()
        plt.savefig(f"../images/{name}_int_grad_{lbl.lower()}.png", dpi=150, bbox_inches="tight")
        plt.show()

# Aggregate Metrics (N=100)

In [ ]:
K = 10

def jaccard_k(scores_a, scores_b, k=K):
    keys = set(scores_a) & set(scores_b)
    if len(keys) < k:
        return float("nan")
    topk_a = set(sorted(keys, key=lambda w: abs(scores_a[w]), reverse=True)[:k])
    topk_b = set(sorted(keys, key=lambda w: abs(scores_b[w]), reverse=True)[:k])
    inter  = len(topk_a & topk_b)
    union  = len(topk_a | topk_b)
    return inter / union if union > 0 else float("nan")

def rank_corr(scores_a, scores_b):
    keys = list(set(scores_a) & set(scores_b))
    if len(keys) < 3:
        return float("nan")
    va = [scores_a[w] for w in keys]
    vb = [scores_b[w] for w in keys]
    rho, _ = spearmanr(va, vb)
    return rho

def pearson_corr(scores_a, scores_b):
    keys = list(set(scores_a) & set(scores_b))
    if len(keys) < 3:
        return float("nan")
    va = [scores_a[w] for w in keys]
    vb = [scores_b[w] for w in keys]
    r, _ = pearsonr(va, vb)
    return r

def sign_agreement_k(scores_a, scores_b, k=K):
    keys = set(scores_a) & set(scores_b)
    if len(keys) < k:
        return float("nan")
    topk_a = sorted(keys, key=lambda w: abs(scores_a[w]), reverse=True)[:k]
    agree  = sum(1 for w in topk_a if w in scores_b and
                 np.sign(scores_a[w]) == np.sign(scores_b[w]))
    return agree / k

def faithfulness_deletion(model, text, scores, k=K, true_class_idx=0):
    if not scores:
        return float("nan")
    words_sorted = sorted(scores, key=lambda w: abs(scores[w]), reverse=True)[:k]
    words_to_mask = set(words_sorted)
    masked = " ".join(
        "[MASK]" if w in words_to_mask else w
        for w in text.split()
    )
    original_prob = classify_fn([text],  model)[0, true_class_idx]
    masked_prob   = classify_fn([masked], model)[0, true_class_idx]
    return float(original_prob - masked_prob)

In [ ]:
# Compute attributions for all models × methods (class-conditioned on true label)
all_attribs = {name: {"lime": [], "shap": [], "ig": []} for name in MODELS}

for name, model in MODELS.items():
    print(f"\n{'='*50}\n{MODEL_LABELS[name]}\n{'='*50}")
    model.to(device)

    for i, text in enumerate(tqdm(eval_texts, desc="  LIME")):
        ci = eval_class_indices[i]
        all_attribs[name]["lime"].append(get_lime_scores(model, text, class_idx=ci))

    for i, text in enumerate(tqdm(eval_texts, desc="  SHAP")):
        ci = eval_class_indices[i]
        all_attribs[name]["shap"].append(get_shap_scores(model, text, class_idx=ci))

    for i, text in enumerate(tqdm(eval_texts, desc="  IG")):
        ci = eval_class_indices[i]
        all_attribs[name]["ig"].append(
            get_ig_scores(model, text, class_token_ids[ci])
        )

    model.cpu()
    torch.cuda.empty_cache()
    print("  Done.")

In [ ]:
teacher_name = "teacher"
rows = []

for method in ["lime", "shap", "ig"]:
    t_attribs = all_attribs[teacher_name][method]
    for name in MODELS:
        s_attribs = all_attribs[name][method]
        model     = MODELS[name]
        model.to(device)

        jaccards, rhos, pearsons, signs, faiths = [], [], [], [], []
        for i in range(len(eval_texts)):
            ta, sa = t_attribs[i], s_attribs[i]
            ci     = eval_class_indices[i]
            jaccards.append(jaccard_k(ta, sa))
            rhos.append(rank_corr(ta, sa))
            pearsons.append(pearson_corr(ta, sa))
            signs.append(sign_agreement_k(ta, sa))
            faiths.append(faithfulness_deletion(model, eval_texts[i], sa,
                                                 true_class_idx=ci))

        model.cpu()
        torch.cuda.empty_cache()
        rows.append({
            "Method":          method.upper(),
            "Model":           MODEL_LABELS[name],
            f"Jaccard@{K}":    round(np.nanmean(jaccards), 4),
            "Spearman ρ":      round(np.nanmean(rhos),     4),
            "Pearson r":       round(np.nanmean(pearsons),  4),
            f"Sign agr@{K}":   round(np.nanmean(signs),    4),
            f"Faithfulness@{K}": round(np.nanmean(faiths), 4),
        })

df_metrics = pd.DataFrame(rows)
for method in ["LIME", "SHAP", "IG"]:
    print(f"\n{'─'*60}\n{method}\n{'─'*60}")
    display(df_metrics[df_metrics["Method"] == method].drop(columns="Method").reset_index(drop=True))

In [ ]:
def stability_lime(model, texts, k=K, M=5):
    scores_per_run = []
    for seed in range(M):
        run = []
        for i, text in enumerate(texts):
            ci = eval_class_indices[i]
            run.append(get_lime_scores(model, text, class_idx=ci, num_samples=2000, seed=seed))
        scores_per_run.append(run)

    stab_vals = []
    for i in range(len(texts)):
        pair_jaccards = []
        for r1 in range(M):
            for r2 in range(r1 + 1, M):
                pair_jaccards.append(
                    jaccard_k(scores_per_run[r1][i], scores_per_run[r2][i], k=k)
                )
        stab_vals.append(np.nanmean(pair_jaccards))
    return np.nanmean(stab_vals)

stab_texts = eval_texts[:20]
print("LIME Stability (N=20, M=5 runs, Jaccard@10):")
for name, model in MODELS.items():
    model.to(device)
    stab = stability_lime(model, stab_texts)
    model.cpu()
    torch.cuda.empty_cache()
    print(f"  {MODEL_LABELS[name]:30s}  {stab:.4f}")

# KL Comparisons

In [ ]:
def compute_avg_kl(teacher, student, dataloader, device, temperature=1.0):
    teacher.eval(); student.eval()
    total_kl, n = 0.0, 0
    with torch.no_grad():
        for batch in dataloader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            last = mask.sum(dim=1) - 1
            idx  = torch.arange(ids.size(0), device=device)
            t_logits = teacher(input_ids=ids, attention_mask=mask).logits[idx, last]
            s_logits = student(input_ids=ids, attention_mask=mask).logits[idx, last]
            t_probs     = F.softmax(t_logits / temperature, dim=-1)
            s_log_probs = F.log_softmax(s_logits / temperature, dim=-1)
            kl = F.kl_div(s_log_probs, t_probs, reduction="batchmean")
            total_kl += kl.item() * ids.size(0)
            n        += ids.size(0)
    return total_kl / n

In [ ]:
teacher = MODELS["teacher"]
teacher.to(device)

print(f"{'Model':<30}  KL (T=1)   KL (T=2)")
print("─" * 50)
for name, model in MODELS.items():
    if name == "teacher":
        continue
    model.to(device)
    kl1 = compute_avg_kl(teacher, model, test_loader, device, temperature=1.0)
    kl2 = compute_avg_kl(teacher, model, test_loader, device, temperature=2.0)
    print(f"{MODEL_LABELS[name]:<30}  {kl1:.4f}     {kl2:.4f}")
    model.cpu()
    torch.cuda.empty_cache()

teacher.cpu()
torch.cuda.empty_cache()